# 01c - Pipeline de pré-processamento

Última parte do pré-processamento (Fase 2 do enunciado). Os notebooks `01_preprocessamento` e `01b_rejeitados` fizeram a **limpeza**: regras fixas, sem nada aprendido com os dados. Aqui constroem-se as **transformações aprendidas** (imputação, *clipping*, padronização, *encoding*), que o enunciado manda ajustar só no treino:

> *"Split before learning transformations. Fit imputers, scalers, encoders, feature selection, PCA, clipping/winsorization (...) only on the training portion of each fold. Use pipelines in both Python and R."*

O que este notebook faz:
1. escolhe as colunas de cada tarefa a partir da **tabela de elegibilidade** da Fase 1;
2. confirma que nenhuma coluna proibida pelo enunciado (*leakage*) entra;
3. define a **validação temporal** (folds com janela expansível);
4. constrói o **pipeline** e mostra, com os dados, que o *fit* só usa o treino de cada fold.

Não se treina nenhum modelo (isso é a Parte 2) e o teste final não é aberto. As *features* novas (≥ 10) são criadas na seleção de modelos; entram neste mesmo pipeline.

**Antes de correr:** é preciso ter corrido `01_preprocessamento.ipynb` (gera `regressao_dev.pkl` e `classificacao_treino.pkl`).

In [ ]:
# Setup: imports e caminhos do projeto
import sys
from pathlib import Path

# Importar o pacote do projeto.
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import pandas as pd

from lending import config
from lending.features import (FORBIDDEN_ALWAYS, FORBIDDEN_PREFIXES, FORBIDDEN_REGRESSION, TASKS,
                              TaskFeatures, forbidden_columns, load_eligibility)
from lending.preprocessing import ExpandingWindowSplit, build_preprocessor
from lending.storage import PickleStore

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 1. Colunas de cada tarefa

A elegibilidade das 151 colunas foi decidida na Fase 1, antes de olhar para os dados (`docs/01_formulacao_problema/elegibilidade_colunas.md`). O código **lê essa tabela diretamente**, em vez de a copiar: se a tabela mudar, o pipeline muda com ela.

Entram no pipeline base as colunas ✅ de cada tarefa, menos:
- as **⚠️ condicionais** (`zip_code`, `addr_state`, `issue_d`; `earliest_cr_line` na regressão): ficam de fora até a condição ser analisada (ex.: a geografia só com uma análise de *fairness*);
- o **texto livre** (`emp_title`, `title`): dezenas de milhares de valores diferentes; só entram como *features* novas;
- as **datas** (`earliest_cr_line` na classificação): só entram transformadas, ex.: antiguidade do crédito = `issue_d − earliest_cr_line`, uma *feature* nova.

As colunas que entram são separadas pelo tratamento que recebem: montantes em USD (com logaritmo), outras numéricas e categorias.

In [ ]:
eligibility = load_eligibility()
print("Tabela de elegibilidade:", eligibility.shape[0], "colunas")
display(eligibility.apply(pd.Series.value_counts).fillna(0).astype(int))

features = {task: TaskFeatures(task, eligibility) for task in TASKS}
pd.DataFrame({task: f.summary() for task, f in features.items()})

In [ ]:
# Categorias de cada tarefa e colunas ✅ que ficam de fora do pipeline base (e porquê)
for task, f in features.items():
    print(f"{task}: categorias = {f.categorical}")

pd.concat({task: f.excluded[f.excluded["tabela"] == "✅"] for task, f in features.items()})

## 2. Verificação contra *leakage*

O enunciado lista colunas que nunca podem ser preditores (desempenho depois da concessão, `hardship_*`, `settlement_*`, ...) e colunas proibidas só na regressão (`grade`, `sub_grade`, `installment`, porque definem ou permitem reconstruir a taxa). Essas listas estão no código (`features.py`) e o `TaskFeatures` **dá erro** se alguma aparecer entre as colunas de uma tarefa. A tabela de elegibilidade já as exclui; esta verificação protege contra um erro na tabela.

In [ ]:
print(f"Proibidas em todas as tarefas: {len(FORBIDDEN_ALWAYS)} colunas + prefixos {FORBIDDEN_PREFIXES}")
print(f"Proibidas só na regressão: {FORBIDDEN_REGRESSION}")

for task, f in features.items():
    assert not forbidden_columns(task, f.columns)
    print(f"{task}: nenhuma coluna proibida entre as {len(f.columns)} de entrada")

# Exemplo da diferença entre tarefas: grade/sub_grade/installment só na classificação
pd.DataFrame({task: {col: col in f.columns for col in ["grade", "sub_grade", "installment", "int_rate"]}
              for task, f in features.items()})

## 3. Validação temporal

A validação respeita o tempo: cada *fold* treina com todos os períodos anteriores e valida no período seguinte (**janela expansível**). Nunca se treina com o futuro de quem se valida, e os empréstimos do mesmo período ficam sempre juntos.

| Tarefa | Dados | Período | *Folds* |
|---|---|---|---|
| Regressão | 2013-01 a 2014-12 | trimestre | 6: o 1.º treina em 2013T1–T2 e valida em 2013T3; o último valida em 2014T4 |
| Classificação | 2007-06 a 2011-12 (36 meses) | ano | 3: validam 2009, 2010 e 2011 |

O teste final (2015) não entra em nenhum *fold*.

In [ ]:
regression = PickleStore.load(config.REG_DEV_PKL, expected_type=pd.DataFrame)
classification = PickleStore.load(config.CLS_TRAIN_PKL, expected_type=pd.DataFrame)

splitters = {"regression": ExpandingWindowSplit("Q", min_train_periods=2),
             "classification": ExpandingWindowSplit("Y", min_train_periods=2)}
data = {"regression": regression, "classification": classification}

for task in splitters:
    print(task)
    display(splitters[task].describe(data[task]["issue_d"]))

## 4. O pipeline

`build_preprocessor` (`src/lending/preprocessing.py`) constrói um `ColumnTransformer` do scikit-learn com três ramos. Tudo o que depende dos dados (entre parênteses) é aprendido no `fit`:

| Ramo | Passos |
|---|---|
| Montantes em USD | colunas vazias no treino saem (quais) → log(1 + x) → *clipping* aos percentis 0,5% e 99,5% (limites) → imputação pela mediana + indicador de "em falta" (medianas) → padronização (média e desvio) |
| Outras numéricas | colunas vazias saem (quais) → *clipping* (limites) → imputação + indicador (medianas) → padronização (média e desvio) |
| Categorias | colunas vazias saem (quais) → "em falta" como categoria → *one-hot* (categorias; as que têm menos de 0,5% do treino ficam juntas) |

**Porquê cada passo:**
- **Logaritmo nos montantes:** rendimentos e saldos são muito assimétricos (ex.: rendimentos de milhões). O log aproxima a distribuição de uma forma simétrica. É uma regra fixa, mas fica no pipeline para ser aplicada sempre na mesma ordem.
- ***Clipping*:** reduz o peso dos valores extremos que a limpeza deixou ficar (ex.: `revol_util` até 892%) sem retirar linhas. Os limites vêm do treino.
- **Imputação com indicador:** o enunciado pede para distinguir "não aplicável" de "em falta" e para não pôr tudo a zero. O indicador regista **que** o valor faltava (ex.: `mths_since_last_delinq` em falta = nunca houve atraso); o valor imputado é a mediana do treino, nunca 0.
- **Colunas vazias:** uma coluna sem valores no treino não tem nada para aprender (ex.: as 37 colunas do *bureau* no treino da classificação). São retiradas também na validação e no teste.
- **Padronização:** põe as variáveis na mesma escala, essencial para o kNN e para as redes neuronais da Parte 2.

In [ ]:
preprocessor = build_preprocessor(features["regression"])
preprocessor

## 5. *Fit* só no treino: demonstração com o *fold* 1 da regressão

O pipeline é ajustado no treino do *fold* 1 (2013T1–T2) e só **aplicado** à validação (2013T3). As verificações mostram, com os dados, que nada da validação entrou no *fit*.

In [ ]:
train_idx, valid_idx = next(splitters["regression"].split(regression["issue_d"]))
cols = features["regression"].columns
X_train, X_valid = regression.iloc[train_idx][cols], regression.iloc[valid_idx][cols]

X_train_t = preprocessor.fit_transform(X_train)
X_valid_t = preprocessor.transform(X_valid)
print(f"Treino: {X_train.shape} -> {X_train_t.shape} | Validação: {X_valid.shape} -> {X_valid_t.shape}")
print(f"Valores em falta depois do pipeline: treino {X_train_t.isna().sum().sum()}, "
      f"validação {X_valid_t.isna().sum().sum()}")

In [ ]:
# 1) As medianas aprendidas são as do treino, não as da validação
imputer = preprocessor.named_transformers_["numeric"]["impute"]
learned = pd.Series(imputer.statistics_, index=imputer.feature_names_in_)
check = ["dti", "fico_range_low", "revol_util", "open_acc"]
medians = pd.DataFrame({"aprendida no fit": learned[check],
                        "mediana do treino": X_train[check].median(),
                        "mediana da validação": X_valid[check].median()})
display(medians)
# (o clipping só mexe nos 0,5% de cada ponta, por isso não muda a mediana)
assert (medians["aprendida no fit"] == medians["mediana do treino"]).all()

# 2) Depois da padronização, a média é 0 e o desvio 1 no treino, mas não na validação
pd.DataFrame({"média treino": X_train_t[check].mean(), "média validação": X_valid_t[check].mean(),
              "desvio treino": X_train_t[check].std(ddof=0), "desvio validação": X_valid_t[check].std(ddof=0)}).round(3)

In [ ]:
# 3) O transform não aprende: aplicar à validação não muda nenhum parâmetro do pipeline
before = imputer.statistics_.copy()
_ = preprocessor.transform(X_valid)
assert (imputer.statistics_ == before).all()

# 4) Mudar a validação não muda o pipeline: ajustar só com o treino dá sempre o mesmo resultado
refit = build_preprocessor(features["regression"]).fit(X_train)
assert refit.transform(X_valid).equals(X_valid_t)
print("O pipeline depende só do treino do fold.")

# Exemplo das colunas criadas: indicadores de "em falta" e one-hot das categorias
indicators = [c for c in X_train_t.columns if c.startswith("missingindicator_")]
print(f"{len(indicators)} indicadores de 'em falta', por exemplo: {indicators[:4]}")
print("one-hot de purpose:", [c for c in X_train_t.columns if c.startswith("purpose_")])

## 6. Classificação: colunas vazias no treino

No treino da classificação (2007–2011), as 37 colunas do *bureau* registadas só a partir de 2012 estão vazias (`docs/02_preprocessamento/valores_em_falta.md`). O passo `DropEmptyColumns` aprende-o no *fit* e retira-as, também na validação.

In [ ]:
train_idx, valid_idx = next(splitters["classification"].split(classification["issue_d"]))
cols = features["classification"].columns
X_train, X_valid = classification.iloc[train_idx][cols], classification.iloc[valid_idx][cols]

classification_preprocessor = build_preprocessor(features["classification"])
X_train_t = classification_preprocessor.fit_transform(X_train)
X_valid_t = classification_preprocessor.transform(X_valid)

dropped = {branch: classification_preprocessor.named_transformers_[branch]["drop_empty"].dropped_
           for branch in ["money", "numeric", "categorical"]}
print(f"Treino: {X_train.shape} -> {X_train_t.shape} | Validação: {X_valid.shape} -> {X_valid_t.shape}")
print(f"Colunas vazias no treino, retiradas: {sum(len(v) for v in dropped.values())}")
print("Por exemplo:", dropped["numeric"][:8])
print("one-hot de grade:", [c for c in X_train_t.columns if c.startswith("grade_")])

## 7. Todos os *folds*

O pipeline é ajustado de novo em cada *fold*, só com o treino desse *fold*. O número de colunas de saída pode mudar de *fold* para *fold* (colunas vazias, categorias raras), e não pode haver valores em falta na saída.

In [ ]:
rows = []
for task in splitters:
    cols = features[task].columns
    for fold, (train_idx, valid_idx) in enumerate(splitters[task].split(data[task]["issue_d"]), start=1):
        pipe = build_preprocessor(features[task])
        X_train_t = pipe.fit_transform(data[task].iloc[train_idx][cols])
        X_valid_t = pipe.transform(data[task].iloc[valid_idx][cols])
        assert X_train_t.notna().all().all() and X_valid_t.notna().all().all()
        rows.append({"tarefa": task, "fold": fold, "linhas treino": len(train_idx),
                     "linhas validação": len(valid_idx), "colunas de saída": X_train_t.shape[1]})
pd.DataFrame(rows).set_index(["tarefa", "fold"])

## 8. Resumo

| O quê | Onde |
|---|---|
| Colunas de cada tarefa | `TaskFeatures` (`src/lending/features.py`), a partir da tabela de elegibilidade |
| Verificação de *leakage* | `forbidden_columns`, com as listas do enunciado |
| Transformações aprendidas | `build_preprocessor` (`src/lending/preprocessing.py`) |
| Validação temporal | `ExpandingWindowSplit` |

**Como se usa na Parte 2:** o pipeline junta-se ao modelo (`Pipeline([("pre", build_preprocessor(...)), ("model", ...)])`) e é passado à validação cruzada com os *folds* de `ExpandingWindowSplit`. Assim, o *fit* do pré-processamento fica sempre dentro do treino de cada *fold*. No fim, ajusta-se com todo o treino e aplica-se (só `transform`) ao teste final.

**Clustering:** usa as mesmas classes (`TaskFeatures("clustering")`), ajustadas no desenvolvimento, que é onde o clustering corre (não há teste nem alvo).